# Pb+Pb 0–10 %: the extra hard scatterings in the hadrons and in unsubtracted jets

The 1,000 minimum-bias events of the GB10 campaign `~/prod/PbPb_0_10_v2x` (Pb+Pb 5.02 TeV,
0–10 %, `--hard none --extra-hard`, extras p̂T > 15 GeV, 265 hydro backgrounds, 40 iSS
oversamples per event). The campaign, its cost and the flow analysis are in
`PBPB_EXTRA_HARD.md` in the campaign's git-ignored run folder `../runs/pbpb_extra_hard/` (`RUN`
below), which also holds this notebook's input and the job records.

Three compositions per oversample k of every event:

| | what | |
|---|---|---|
| **bulk** | `bulk_bg`, sample k of the event's background | no extras at all |
| **bulk + wakes** | `bulk_jet`, sample k of the jet leg | the extras' energy deposited in MUSIC_2 and frozen out |
| **bulk + extras** | `bulk_jet` + `extra_frag` (fragmentation k) | plus the extras' own (not thermalized) hadrons |

`bulk_jet` and `bulk_bg` were sampled correlated (`--correlated`): sample k of both comes from
the same random numbers, so their ratios are much more precise than either spectrum.

The per-event histograms come from `extra_hard_hists.py` (one pass over all events, 7 min with
13 processes) → `RUN/data/extra_hard_hists.npz`. Every histogram is the mean over an event's
oversamples (per event = per oversample), averaged over the events. **Errors: delete-one-block
jackknife over the hydro backgrounds** (events sharing a background are left out together;
265 backgrounds in 100 blocks), for ratios too.

**Contents:** 1. the extras (number, p̂T, initiators) · 2. η and φ of charged hadrons ·
3. charged-hadron pT spectra · 4. anti-kT R = 0.4 jets, unsubtracted · 5. summary

In [ ]:
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

RUN = os.path.join("..", "runs", "pbpb_extra_hard")      # the campaign's git-ignored run folder
d = dict(np.load(os.path.join(RUN, "data", "extra_hard_hists.npz")))
N = len(d["seed"])
for q in ("jet_ch", "jet_all", "lead"):          # the jet histograms share one binning
    d[f"edges_{q}"] = d["edges_jet"]

# fixed series colours (validated categorical slots 1-4) and line styles as the second cue
STYLE = {
    "bg":    dict(color="#2a78d6", ls="-",  label="bulk"),
    "jet":   dict(color="#eb6834", ls="--", label="bulk + wakes"),
    "full":  dict(color="#1baf7a", ls="-.", label="bulk + extras (wakes + extra_frag)"),
    "extra": dict(color="#eda100", ls=":",  label="extra_frag alone"),
}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})

# jackknife over the hydro backgrounds (as analysis/bulk/flow.py): at most 100 blocks
u, inv = np.unique(d["group"], return_inverse=True)
blk_of_group = np.zeros(len(u), int)
for i, part in enumerate(np.array_split(np.arange(len(u)), min(100, len(u)))):
    blk_of_group[part] = i
BLOCK = blk_of_group[inv]
NB = BLOCK.max() + 1

def jk(f, *arrays):
    # f(*means) and its jackknife error; arrays are per event (N, ...)
    val = f(*[a.mean(0) for a in arrays])
    reps = np.array([f(*[a[BLOCK != b].mean(0) for a in arrays]) for b in range(NB)])
    err = np.sqrt((NB - 1) / NB * ((reps - reps.mean(0)) ** 2).sum(0))
    return val, err

def comp(name, q):
    # per-event histograms of one composition; 'extra' = full - jet
    if name == "extra":
        return d[f"full.{q}"] - d[f"jet.{q}"]
    return d[f"{name}.{q}"]

def centers(e):
    return 0.5 * (e[1:] + e[:-1])

def ratio(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(b > 0, a / b, np.nan)

# the extras' cross sections, per job (warm-up of each job's own Pythia)
SIG = {}
for p in glob.glob(os.path.join(RUN, "jobs", "*_seed*.json")):
    j = json.load(open(p))
    SIG[j["seed"]] = (j["extra_sigma"], j["extra_sigma_nn"])
lam = np.array([d["n_coll"][i] * SIG[s][0] / SIG[s][1] for i, s in enumerate(d["seed"])])
print(f"{N} events, {len(u)} backgrounds in {NB} jackknife blocks, "
      f"{int(d['n_samples'].mean())} oversamples per event")

## 1. The extras

Per event N ~ Poisson(λ), λ = N_coll σ(p̂T > 15 GeV) / σ_NN, with each job's warm-up σ
(0.52 mb ± 1 %) and σ_NN = 67.6 mb; vertices uniform over the binary collisions. The expected
distribution of N is the Poisson mixture over the events' λ (N_coll varies 1117–2149 in
0–10 %). Each scattering gives its two outgoing partons plus the ISR partons (Pythia's ISR
on, MPI off): the **initiators**, the partons that start a shower (Matter → LBT).

In [ ]:
n = d["n_extra"]; ni = d["n_initiators"]
print(f"extras per event: mean {n.mean():.2f}, variance {n.var():.2f}  "
      f"(expected: mean {lam.mean():.2f}, variance {lam.mean() + lam.var():.2f} = <λ> + var λ)")
print(f"initiators per event: mean {ni.mean():.1f}, per scattering {ni.sum() / n.sum():.2f};  "
      f"p̂T: mean {d['extra_pthat'].mean():.1f} GeV, max {d['extra_pthat'].max():.0f} GeV;  "
      f"initiator pT max {d['ini_pt'].max():.0f} GeV")

fig, ax = plt.subplots(2, 3, figsize=(13, 7.2))
# (a) N per event vs the Poisson mixture
k = np.arange(0, n.max() + 4)
expect = poisson.pmf(k[:, None], lam[None, :]).mean(1) * N
a = ax[0, 0]
a.bar(k, np.bincount(n, minlength=len(k))[:len(k)], width=0.8, color=STYLE["bg"]["color"],
      alpha=0.8, label="campaign")
a.plot(k, expect, "o", color="#2b2b2b", ms=4, label="Poisson(λ), mixed over the events")
a.set(xlabel="extra hard scatterings per event", ylabel="events"); a.legend(loc="upper right", fontsize=8)
# (b) N vs N_coll
a = ax[0, 1]
a.scatter(d["n_coll"], n, s=6, alpha=0.35, color=STYLE["bg"]["color"], label="events")
eb = np.linspace(d["n_coll"].min(), d["n_coll"].max() + 1, 9)
idx = np.digitize(d["n_coll"], eb) - 1
prof = [n[idx == i].mean() for i in range(len(eb) - 1)]
a.plot(centers(eb), prof, "o-", color="#2b2b2b", ms=5, label="mean per N_coll bin")
xs = np.linspace(eb[0], eb[-1], 10)
a.plot(xs, xs * np.mean([s[0] / s[1] for s in SIG.values()]), color="#eb6834", ls="--",
       label="N_coll σ/σ_NN")
a.set(xlabel="N_coll", ylabel="extras per event"); a.legend(loc="upper left", fontsize=8)
# (c) initiators per event
a = ax[0, 2]
a.hist(ni, bins=np.arange(0, ni.max() + 5, 4), color=STYLE["bg"]["color"], alpha=0.8)
a.set(xlabel="initiators of the extras per event", ylabel="events")
# (d) pTHat spectrum
a = ax[1, 0]
pb = np.arange(15, 121, 3)
h, _ = np.histogram(d["extra_pthat"], pb)
a.errorbar(centers(pb), h / N / np.diff(pb), np.sqrt(h) / N / np.diff(pb), fmt="o", ms=4,
           color=STYLE["bg"]["color"])
a.set(yscale="log", xlabel=r"$\hat p_T$ of the extras [GeV]",
      ylabel=r"$dN/d\hat p_T$ per event [1/GeV]")
# (e) initiator pT spectrum
a = ax[1, 1]
ib = np.concatenate([np.arange(0, 20, 1.0), np.arange(20, 60, 4.0), np.arange(60, 141, 10.0)])
h, _ = np.histogram(d["ini_pt"], ib)
a.errorbar(centers(ib), h / N / np.diff(ib), np.sqrt(h) / N / np.diff(ib), fmt="o", ms=4,
           color=STYLE["bg"]["color"])
a.set(yscale="log", xlabel="initiator $p_T$ [GeV]", ylabel="$dN/dp_T$ per event [1/GeV]")
# (f) initiator eta
a = ax[1, 2]
eb2 = np.linspace(-8, 8, 41)
for cut, st in ((0, "bg"), (10, "jet")):
    m = d["ini_pt"] > cut
    h, _ = np.histogram(d["ini_eta"][m], eb2)
    a.step(centers(eb2), h / N / np.diff(eb2), where="mid", color=STYLE[st]["color"],
           ls=STYLE[st]["ls"], label=f"initiators, $p_T$ > {cut} GeV")
a.axvspan(-2.4, 2.4, color="#888", alpha=0.12, label="hadron acceptance |η| < 2.4")
a.set(xlabel="initiator η", ylabel="dN/dη per event"); a.legend(fontsize=8)
for x, l in zip(ax.flat, "abcdef"):
    x.set_title(f"({l})", loc="left", fontsize=10)
fig.tight_layout()

## 2. η and φ of charged hadrons

Charged hadrons with pT > 0.15 GeV; η over the files' reach |η| < 2.6, φ in |η| < 2.4. The
lab φ is not flat: the impact parameter has a fixed direction in the lab, so dN/dφ carries the
event's elliptic flow. Against the background's own Ψ₂ (from all its oversamples, charged
0.2 < pT < 3 GeV, |η| < 2.4) the modulation is that of v₂ relative to Ψ₂; the same Ψ₂ is used
for all three compositions. (The hadrons that determine Ψ₂ are among those histogrammed, so
the modulation itself is biased up by autocorrelation, equally for all three; the ratios are
not affected at this level.)

In [ ]:
def spectrum_with_ratios(q, xlabel, ylabel, comps=("bg", "jet", "full"), logy=False,
                         ratios=(("jet", "bg"), ("full", "bg")), xlim=None, rlim=None,
                         title=None):
    e = d[f"edges_{q}"]; c = centers(e); w = np.diff(e)
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(7, 6.2), sharex=True,
                                 gridspec_kw=dict(height_ratios=[2, 1.3], hspace=0.06))
    for nm in comps:
        v, err = jk(lambda x: x / w, comp(nm, q))
        a0.errorbar(c, v, err, drawstyle="steps-mid", **STYLE[nm])
    a0.set(ylabel=ylabel, yscale="log" if logy else "linear")
    a0.legend(fontsize=8)
    if title:
        a0.set_title(title, fontsize=10)
    for num, den in ratios:
        v, err = jk(ratio, comp(num, q), comp(den, q))
        st = dict(STYLE[num]); st["label"] = f"{STYLE[num]['label']} / {STYLE[den]['label']}"
        a1.errorbar(c, v, err, fmt="o", ms=3.5, capsize=0, color=st["color"], label=st["label"])
    a1.axhline(1, color="#555", lw=1)
    a1.set(xlabel=xlabel, ylabel="ratio")
    if rlim:
        a1.set_ylim(*rlim)
    if xlim:
        a1.set_xlim(*xlim)
    a1.legend(fontsize=8)
    return fig

spectrum_with_ratios("eta", "η", "dN/dη (charged, $p_T$ > 0.15 GeV)",
                     title="charged hadrons, all events");

In [ ]:
spectrum_with_ratios("phi", "φ (lab) [rad]", "dN/dφ (charged, |η| < 2.4)");
fig = spectrum_with_ratios("dphi2", "φ − Ψ₂ (mod π) [rad]", "dN/dφ (charged, |η| < 2.4)")

# the ratio's modulation: R(Δφ) = a (1 + 2 b cos 2Δφ), b ≈ Δv₂ relative to Ψ₂
e = d["edges_dphi2"]; c = centers(e)
def fit_mod(num, den):
    r = ratio(num, den); A = np.vstack([np.ones_like(c), np.cos(2 * c)]).T
    a, b2 = np.linalg.lstsq(A, r, rcond=None)[0]
    return np.array([a, b2 / (2 * a)])
for nm in ("jet", "full"):
    (a, b), (ea, eb_) = jk(fit_mod, comp(nm, "dphi2"), comp("bg", "dphi2"))
    v2bg, ev2 = jk(lambda h: (h * np.cos(2 * c)).sum() / h.sum(), comp("bg", "dphi2"))
    print(f"{STYLE[nm]['label']:>35s} / bulk:  a = {a:.4f} ± {ea:.4f},  "
          f"2b cos2Δφ amplitude: b = {b:+.5f} ± {eb_:.5f}  (bulk's <cos 2(φ−Ψ₂)> = {v2bg:.4f})")

## 3. Charged-hadron pT spectra, |η| < 0.8

Where the extras' own hadrons (`extra_frag`, not thermalized) take over from the bulk.

In [ ]:
fig = spectrum_with_ratios("pt", "$p_T$ [GeV]", "dN/d$p_T$ per event [1/GeV]",
                           comps=("bg", "jet", "full", "extra"), logy=True, rlim=(0.9, 200))
fig.axes[0].set_xscale("log"); fig.axes[1].set_xscale("log"); fig.axes[1].set_yscale("log")
e = d["edges_pt"]; c = centers(e)
fe, _ = jk(ratio, comp("extra", "pt"), comp("full", "pt"))
cross = c[np.argmax(fe > 0.5)] if (fe > 0.5).any() else None
print("fraction of extra_frag in the full event's charged hadrons:")
for lo in (1, 2, 4, 6, 10, 20):
    i = np.searchsorted(e, lo) ; print(f"   pT ≈ {c[i]:5.1f} GeV: {fe[i]:.3f}")
print(f"extra_frag is more than half of the hadrons above pT ≈ {cross:.1f} GeV" if cross else "")

## 4. Jets: anti-kT R = 0.4, unsubtracted

FastJet anti-kT R = 0.4 on each oversample, **no background subtraction**: constituents pT >
0.15 GeV, |η| < 2.4; jets |η| < 2.0. Charged-particle jets (charged hadrons) and jets of
all hadrons (charged and neutral; in our decays π⁰ and η are stable, so they enter as single
particles). In central Pb+Pb an unsubtracted R = 0.4 jet is mostly bulk: its spectrum peaks
near the bulk's ρ·πR². The question is
whether the extras show up in the tails beyond the bulk's fluctuations, and how much of
that comes from the wakes (thermalized) versus the extras' own fragments.

Spectra are per event (per oversample) per GeV: inclusive jets, and the leading jet of each
oversample.

In [ ]:
XMAX = {"ch": 270, "all": 410}
for kind, nm in (("ch", "charged-particle jets"), ("all", "jets of all hadrons")):
    fig = spectrum_with_ratios(f"jet_{kind}", "jet $p_T$ (unsubtracted) [GeV]",
                               "dN/d$p_T$ per event [1/GeV]", logy=True, rlim=(0.5, 20),
                               xlim=(0, XMAX[kind]),
                               title=f"inclusive {nm}, R = 0.4, |η_jet| < 2.0")
    fig.axes[1].set_yscale("log")

In [ ]:
for kind, nm in (("ch", "charged-particle jets"), ("all", "jets of all hadrons")):
    for c_ in ("bg", "jet", "full"):
        d[f"{c_}.lead"] = d[f"{c_}.lead_{kind}"]
    fig = spectrum_with_ratios("lead", "leading jet $p_T$ (unsubtracted) [GeV]",
                               "probability density [1/GeV]", logy=True, rlim=(0.3, 30),
                               xlim=(50, XMAX[kind]),
                               title=f"leading {nm} per oversample, R = 0.4")
    fig.axes[1].set_yscale("log")

In [ ]:
# the tails: probability that an oversample's leading jet is above a threshold
e = d["edges_jet"]
def above(h, x):
    i = np.searchsorted(e, x); return h[..., i:].sum(-1)
rows = []
for kind, ths in (("ch", (120, 150, 168, 186)), ("all", (240, 270, 300, 318))):   # bin edges
    for x in ths:
        vals = {c_: jk(lambda h: above(h, x), d[f"{c_}.lead_{kind}"]) for c_ in ("bg", "jet", "full")}
        rj, erj = jk(lambda a, b: above(a, x) / above(b, x), d[f"jet.lead_{kind}"], d[f"bg.lead_{kind}"])
        rf, erf = jk(lambda a, b: above(a, x) / above(b, x), d[f"full.lead_{kind}"], d[f"bg.lead_{kind}"])
        rows.append((kind, x, *vals["bg"], *vals["full"], rj, erj, rf, erf))
print(f"{'jets':>4s} {'> pT':>5s} {'P(bulk)':>18s} {'P(bulk + extras)':>20s} {'wakes / bulk':>14s} {'extras / bulk':>15s}")
for k_, x, pb, epb, pf, epf, rj, erj, rf, erf in rows:
    print(f"{k_:>4s} {x:5.0f} {pb:9.2e} ± {epb:7.1e} {pf:10.2e} ± {epf:7.1e} "
          f"{rj:6.2f} ± {erj:4.2f} {rf:7.2f} ± {erf:4.2f}")

## 5. Summary

The results and their discussion are kept with the campaign, in the git-ignored run folder
`../runs/pbpb_extra_hard/`: the executed notebook there and `PBPB_EXTRA_HARD.md`.